# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
#loading the data
!pip install -q duckdb huggingface_hub

import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
base = "hf://datasets/FlyRank/internship-warehouse"

def load_month_agg(month_str):
    return con.sql(f"""
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS gsc_impressions,
               SUM(gsc_clicks) AS gsc_clicks,
               SUM(gsc_sum_position) AS gsc_sum_position,
               SUM(ga4_sessions) AS ga4_sessions,
               SUM(ga4_engaged_sessions) AS ga4_engaged_sessions
        FROM read_parquet('{base}/fact_content_daily_performance/month={month_str}/data_0.parquet')
        GROUP BY client_hash_id, content_hash_id
    """).df()

df_feb_agg = load_month_agg('2026-02')
df_march_agg = load_month_agg('2026-03')

df_april_agg = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS gsc_clicks_apr
    FROM read_parquet('{base}/fact_content_daily_performance/month=2026-04/data_0.parquet')
    GROUP BY client_hash_id, content_hash_id
""").df()

print(f"Feb: {len(df_feb_agg)}, March: {len(df_march_agg)}, April(clicks only): {len(df_april_agg)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feb: 321546, March: 331437, April(clicks only): 362172


In [2]:
# rule code from previous weeks
def position_bucket(pos):
    if pd.isna(pos):
        return 'no_position_data'
    elif pos <= 3:
        return '1-3 (top)'
    elif pos <= 10:
        return '4-10'
    elif pos <= 20:
        return '11-20'
    else:
        return '21+'

def build_rule_df(raw_agg):
    d = raw_agg.copy()
    d['gsc_avg_position'] = d['gsc_sum_position'] / d['gsc_impressions']
    d.loc[d['gsc_sum_position'] == 0, 'gsc_avg_position'] = pd.NA

    signal = d[d['gsc_impressions'] > 0].copy()
    signal['ctr'] = signal['gsc_clicks'] / signal['gsc_impressions']
    signal['position_bucket'] = signal['gsc_avg_position'].apply(position_bucket)
    peer_ctr = signal.groupby('position_bucket')['ctr'].mean().to_dict()

    d['ctr'] = d['gsc_clicks'] / d['gsc_impressions']
    d['engagement_rate'] = d['ga4_engaged_sessions'] / d['ga4_sessions']
    d['position_bucket'] = d['gsc_avg_position'].apply(position_bucket)
    d['peer_avg_ctr'] = d['position_bucket'].map(peer_ctr)

    conditions = [
        (d['gsc_impressions'] >= 50) & d['ctr'].notna() & d['peer_avg_ctr'].notna()
            & (d['ctr'] < 0.5 * d['peer_avg_ctr']),
        (d['ga4_sessions'] >= 10) & d['engagement_rate'].notna()
            & (d['engagement_rate'] < 0.10),
    ]
    d['action'] = np.select(conditions, ['snippet_fix', 'content_fix'], default='monitor')
    d['reason_code'] = np.select(conditions,
        ['CTR_below_half_position_peers', 'engagement_below_10pct_reliable'], default='no_flag_triggered')
    d['rule_score'] = np.select(conditions,
        [(d['peer_avg_ctr'] - d['ctr']) * d['gsc_impressions'],
         d['ga4_sessions'] * (0.10 - d['engagement_rate']) * 10], default=0)

    eligible = (d['gsc_impressions'] >= 50) | (d['ga4_sessions'] >= 10)
    return d[eligible].copy()

march_rule_df = build_rule_df(df_march_agg)
feb_rule_df = build_rule_df(df_feb_agg)
print(f"March-eligible: {len(march_rule_df)}, Feb-eligible: {len(feb_rule_df)}")
print(march_rule_df['action'].value_counts())

March-eligible: 116512, Feb-eligible: 93871
action
snippet_fix    75663
monitor        28755
content_fix    12094
Name: count, dtype: int64


In [3]:
#decline label
MIN_CLICKS_FOR_LABEL = 5
DECLINE_THRESHOLD = 0.8

def build_labelable(rule_df, next_month_clicks_df, next_month_col):
    merged = rule_df.merge(next_month_clicks_df, on=['client_hash_id', 'content_hash_id'], how='left')
    tracked = merged[next_month_col].notna()
    lab = merged[tracked & (merged['gsc_clicks'] >= MIN_CLICKS_FOR_LABEL)].copy()
    lab['decline_label'] = (lab[next_month_col] < DECLINE_THRESHOLD * lab['gsc_clicks']).astype(int)
    return lab

labelable = build_labelable(march_rule_df, df_april_agg, 'gsc_clicks_apr')
print(f"March->April labelable: {len(labelable)}, base rate {labelable['decline_label'].mean():.3f}")

FEATURES = ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions',
            'ga4_engaged_sessions', 'ctr', 'engagement_rate', 'peer_avg_ctr']

def build_X(df, feature_cols):
    X = df[feature_cols].copy()
    worst = X['gsc_avg_position'].max()
    X['gsc_avg_position'] = X['gsc_avg_position'].fillna(worst + 10 if pd.notna(worst) else 100)
    X['ga4_sessions'] = X['ga4_sessions'].fillna(0)
    X['ga4_engaged_sessions'] = X['ga4_engaged_sessions'].fillna(0)
    X['engagement_rate'] = X['engagement_rate'].fillna(0)
    X['ctr'] = X['ctr'].fillna(0)
    X['peer_avg_ctr'] = X['peer_avg_ctr'].fillna(X['peer_avg_ctr'].median())
    X = X.join(pd.get_dummies(df['position_bucket'], prefix='pos', drop_first=True))
    return X

March->April labelable: 28805, base rate 0.545


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

## Top-10 review

**1. `quiet_riser` — investigate_quiet_risk** — 244,931 impressions, 669 clicks (CTR 0.27%), position ~15.2 (page 2), engagement 11.6% (just above the 10% rule threshold, so `content_fix` never triggered). *Why flagged:* the model's highest score in the entire queue (0.92), despite the rule seeing nothing wrong — this is exactly the "rule missed it" case the archetype is built to catch. *Wrong if:* the page is a genuinely borderline case sitting just above the rule's hard 10% engagement cutoff — a small shift either way and the rule would have caught it too, so the model may just be sensitive near that boundary rather than seeing something the rule structurally can't.

**2. `confirmed_risk` — review_before_revert** — 212,404 impressions, only 24 clicks, position **0.67 (data-quality flag — see w04)**. *Why:* near-zero CTR at a claimed top position, both signals agree. *Wrong if:* acted on before the position value is verified against raw daily data — this is the same unreliable-position issue from w04, now resurfacing with a high model score attached to it.

**3. `quiet_riser` — investigate_quiet_risk** — only 1,322 impressions, 24 clicks, decent engagement (21.8%). *Why:* small-traffic page the model still rated high risk. *Wrong if:* the sample is just too small for either signal to be trustworthy — 1,322 impressions is tiny next to row 1's 244,931.

**4. `quiet_riser` — investigate_quiet_risk** — 467 impressions, 1 click, position ~16. *Why:* similar to #3, small and quiet but model-flagged. *Wrong if:* same small-sample concern as #3 — worth a combined note rather than treating each separately.

**5. `confirmed_risk` — review_before_revert** — 134,984 impressions, 1 click, position ~2.7 (near top of page 1). *Why:* extreme click failure at a strong ranking — classic "shop window" pattern from w04. *Wrong if:* a tracking/measurement error rather than real user behavior, given how extreme the gap is.

**6. `confirmed_risk` — review_before_revert** — 124,075 impressions, 1 click, position **0.31 (data-quality flag)**. *Why:* same near-zero-click pattern as #5. *Wrong if:* not verified against raw data first — same caveat as #2.

**7. `confirmed_risk` — review_before_revert** — 186,983 impressions, 586 clicks, position ~2.4. *Why:* CTR still below peer average despite a strong position. *Wrong if:* seasonal/topical timing is temporarily suppressing clicks rather than a real snippet problem (per w04's "check the calendar first" caveat).

**8. `confirmed_risk` — review_before_revert** — 143,019 impressions, 43 clicks, position ~3.2, session data missing (`NaN`). *Why:* CTR-based flag, model agrees. *Wrong if:* the missing GA4 session data means engagement can't actually be verified — this pick is judged on CTR alone.

**9. `confirmed_risk` — review_before_revert** — 83,834 impressions, 1 click, position **0.12 (data-quality flag — the most extreme of the three)**. *Why:* same shop-window pattern as #2/#6. *Wrong if:* not verified against raw data — this is the least trustworthy position value in the whole top 10.

**10. `confirmed_risk` — review_before_revert** — 203,497 impressions, 289 clicks, position ~2.5. *Why:* CTR below peers at a strong position, moderate engagement. *Wrong if:* this client's baseline CTR is naturally lower than the peer average for reasons unrelated to the snippet (niche topic, technical audience).

**Pattern worth flagging:** 3 of these 10 picks (#2, #6, #9) carry the same sub-1.0 position data-quality issue first documented in w04 — and it's now showing up inside the validated model's own highest-confidence picks, not just the rule's. That's a stronger version of the same caveat, and it carries forward into Sections 2/3 rather than being old news.

In [4]:
#model
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

X_train = build_X(labelable, FEATURES)
y_train = labelable['decline_label'].values

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

deployed_model = LogisticRegression(max_iter=1000, random_state=0)
deployed_model.fit(X_train_scaled, y_train)

# score every March-eligible page, not just the labelable subset
X_all = build_X(march_rule_df, FEATURES)
X_all = X_all.reindex(columns=X_train.columns, fill_value=0)
X_all_scaled = scaler.transform(X_all)
march_rule_df['logreg_prob'] = deployed_model.predict_proba(X_all_scaled)[:, 1]

print(f"Scored {len(march_rule_df)} eligible pages")
print(f"  ({len(labelable)} of those had a known April outcome and were used to fit the model)")
march_rule_df['logreg_prob'].describe()

Scored 116512 eligible pages
  (28805 of those had a known April outcome and were used to fit the model)


,logreg_prob
count,116512.000000
mean,0.541080
std,0.076669
min,0.000002
25%,0.523094
50%,0.565046
75%,0.585316
max,0.922834


In [5]:
HIGH_RISK_CUTOFF = march_rule_df['logreg_prob'].quantile(0.90) # 90 %ile (page at high risk of decline as per model)
print(f"Top-10% cutoff: {HIGH_RISK_CUTOFF:.4f}")
print(f"Pages at/above it: {(march_rule_df['logreg_prob'] >= HIGH_RISK_CUTOFF).sum()}")

Top-10% cutoff: 0.6084
Pages at/above it: 11652


In [6]:
def assign_archetype(row):
    flagged = row['action'] != 'monitor'
    high_risk = row['logreg_prob'] >= HIGH_RISK_CUTOFF
    if flagged and high_risk:
        return pd.Series(['confirmed_risk', 'review_before_revert'])
    elif (not flagged) and high_risk:
        return pd.Series(['quiet_riser', 'investigate_quiet_risk'])
    elif flagged and not high_risk:
        return pd.Series(['flagged_lower_urgency', 'verify_then_review'])
    else:
        return pd.Series(['stable', 'monitor_only'])

march_rule_df[['archetype', 'playbook_action']] = march_rule_df.apply(assign_archetype, axis=1)

print(march_rule_df['archetype'].value_counts())
print()
print(march_rule_df['playbook_action'].value_counts())
print()
print("Cross-tab (sanity check that the mapping did what the table says):")
print(pd.crosstab(march_rule_df['archetype'], march_rule_df['action']))

archetype
flagged_lower_urgency    76547
stable                   28313
confirmed_risk           11210
quiet_riser                442
Name: count, dtype: int64

playbook_action
verify_then_review        76547
monitor_only              28313
review_before_revert      11210
investigate_quiet_risk      442
Name: count, dtype: int64

Cross-tab (sanity check that the mapping did what the table says):
action                 content_fix  monitor  snippet_fix
archetype                                               
confirmed_risk                  79        0        11131
flagged_lower_urgency        12015        0        64532
quiet_riser                      0      442            0
stable                           0    28313            0


In [7]:
feb_labelable = build_labelable(
    feb_rule_df,
    df_march_agg[['client_hash_id', 'content_hash_id', 'gsc_clicks']].rename(columns={'gsc_clicks': 'gsc_clicks_mar'}),
    'gsc_clicks_mar'
)

print(f"Feb->March labelable:   {len(feb_labelable)}, base rate {feb_labelable['decline_label'].mean():.3f}")
print(f"March->April labelable: {len(labelable)}, base rate {labelable['decline_label'].mean():.3f}")
print(f"Shift: {labelable['decline_label'].mean() - feb_labelable['decline_label'].mean():+.3f} percentage points")

Feb->March labelable:   21773, base rate 0.313
March->April labelable: 28805, base rate 0.545
Shift: +0.232 percentage points


In [8]:
ranked_queue = march_rule_df.sort_values('logreg_prob', ascending=False).reset_index(drop=True)

review_cols = ['client_hash_id', 'content_hash_id', 'archetype', 'playbook_action',
               'action', 'reason_code', 'logreg_prob', 'rule_score',
               'gsc_impressions', 'gsc_clicks', 'ctr', 'gsc_avg_position',
               'ga4_sessions', 'engagement_rate']

print("Top 10 of the ranked queue:")
ranked_queue[review_cols].head(10)

Top 10 of the ranked queue:


,client_hash_id,content_hash_id,archetype,playbook_action,action,reason_code,logreg_prob,rule_score,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,ga4_sessions,engagement_rate
0,client_23a62021009f63c4,content_e8a52cf3d5988c07,quiet_riser,investigate_quiet_risk,monitor,no_flag_triggered,0.922834,0.000000,244931.0,669.0,0.002731,15.173490,891.0,0.115600
1,client_23a62021009f63c4,content_44f34c0a90047651,confirmed_risk,review_before_revert,snippet_fix,CTR_below_half_position_peers,0.870913,2091.787368,212404.0,24.0,0.000113,0.665877,37.0,0.027027
2,client_9958f0a7ae1df715,content_01ab236521ddf2f9,quiet_riser,investigate_quiet_risk,monitor,no_flag_triggered,0.820197,0.000000,1322.0,24.0,0.018154,24.565809,229.0,0.218341
3,client_fef1a8f436438636,content_c30950c89c368054,quiet_riser,investigate_quiet_risk,monitor,no_flag_triggered,0.818633,0.000000,467.0,1.0,0.002141,15.995717,167.0,0.203593
4,client_73cda7b4e4f265ea,content_8e1334d6356668e3,confirmed_risk,review_before_revert,snippet_fix,CTR_below_half_position_peers,0.805959,1343.595403,134984.0,1.0,0.000007,2.693038,4.0,0.000000
5,client_73cda7b4e4f265ea,content_fec55986a1868d62,confirmed_risk,review_before_revert,snippet_fix,CTR_below_half_position_peers,0.802826,1234.929256,124075.0,1.0,0.000008,0.308426,0.0,NaN
6,client_e547b89c05043229,content_4ffe18112a5642e3,confirmed_risk,review_before_revert,snippet_fix,CTR_below_half_position_peers,0.783712,1276.565062,186983.0,586.0,0.003134,2.389966,364.0,0.164835
7,client_62f4a7e64f5e0096,content_34a70fea29d15f24,confirmed_risk,review_before_revert,snippet_fix,CTR_below_half_position_peers,0.768696,653.906389,143019.0,43.0,0.000301,3.166132,NaN,NaN
8,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,confirmed_risk,review_before_revert,snippet_fix,CTR_below_half_position_peers,0.754714,834.082758,83834.0,1.0,0.000012,0.116003,0.0,NaN
9,client_e547b89c05043229,content_8d7d99f109e19aa2,confirmed_risk,review_before_revert,snippet_fix,CTR_below_half_position_peers,0.751345,1738.063435,203497.0,289.0,0.001420,2.468557,164.0,0.097561


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*


**Who this is for:** a reviewer with a fixed weekly review capacity, deciding which pages to
look at first — not deciding what change to make once they get there.

**Scope:** the March-2026 eligible population only (116,512 pages meeting ≥50 GSC impressions
OR ≥10 GA4 sessions). The queue says nothing about pages outside this filter, and nothing about
any month other than the one it was scored on.

**Where it stops being valid:**

- **Precision is 0.58–0.60, not 0.90.** That's the honest, client-grouped and time-aware number
  from `w06` — the number this deployed model should be judged against. At that precision,
  roughly 40% of pages ranked as risky will not actually decline; this queue orders review
  priority, it does not confirm outcomes.
- **This queue reflects one month's decline rate, which is not stable across months.** The
  Feb→Mar and Mar→Apr base rates moved from 0.313 to 0.545 — a +0.232 shift between two
  adjacent windows using the identical rule and label definition. A queue built this cycle
  should not be assumed to carry the same base rate next cycle (full discussion in Section 4).
- **`HIGH_RISK_CUTOFF` (0.6084) is a population-relative cutoff, not a fixed risk scale.** It
  marks the top 10% of *this* March population. Re-running the same code on a different month
  will produce a different cutoff value even if nothing about underlying page risk changed —
  the number should be recomputed each cycle, never hardcoded.
- **Pages with no recorded search position (`no_position_data`) are scored without that signal.**
  The training population contained none of this category, so the model has no learned
  coefficient for it — its score for these pages should be trusted less than a normal page's.
- **A known position data-quality issue (first flagged in `w04`) reaches into the model's own
  top picks.** 3 of the top 10 confirmed-risk pages carry a `gsc_avg_position` below 1 — a
  value Google cannot actually produce. This isn't just a rule artifact anymore; it's inside
  the validated model's highest-confidence output too.
- **Two-thirds of the eligible population (76,547 of 116,512 pages) lands in
  `verify_then_review`.** The queue is sharp at its extremes (`confirmed_risk`,
  `quiet_riser`) but the middle of the distribution doesn't discriminate much — most pages get
  one broad, moderate-priority label rather than a decisive signal.
- **This is decision-support, not a causal claim.** A `review_before_revert` label means a page
  matches a pattern worth a reviewer's time — it does not mean the page is broken, and fixing
  it is not guaranteed to reverse any decline.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Per-archetype review checklist:**

- **`confirmed_risk` (review_before_revert):** before acting, verify `gsc_avg_position` against
  raw daily data if it's below 1 — a known data-quality artifact (Section 1) that shows up in
  3 of the top 10 picks in this archetype. Also check for a seasonal/campaign explanation before
  assuming a genuine snippet or content problem.
- **`quiet_riser` (investigate_quiet_risk):** check how close the page sits to the rule's own
  thresholds (CTR < 50% of peer average, engagement < 10%) before treating this as a real blind
  spot the rule structurally can't see. The top-ranked page in this notebook's run sat at 11.6%
  engagement — just above the rule's 10% line — which may mean the model is sensitive near a
  boundary, not detecting something categorically different.
- **`flagged_lower_urgency` (verify_then_review):** this archetype is two-thirds of the entire
  queue (Section 2). Before treating any individual pick here as a priority, confirm it isn't
  simply riding the rule's baseline sensitivity — a human should sample this group, not
  work it top-to-bottom like the smaller archetypes.
- **`stable` (monitor_only):** no review required this cycle.

**The no-go list — what this playbook must never do on its own:**

- It must never auto-edit, auto-revert, or auto-publish any content change. Every action name
  (`review_before_revert`, `verify_then_review`, `investigate_quiet_risk`) starts with a human
  verb on purpose — the queue tells a person where to look, it does not act.
- `HIGH_RISK_CUTOFF` must never be hardcoded from a past run. It's recomputed from the live
  population each cycle (Section 2) — reusing an old cutoff value would silently misclassify
  every page as the underlying score distribution shifts.
- A `logreg_prob` score must never be presented to a reviewer without its matching `action` /
  `reason_code` from the rule alongside it. The score says *how urgent*; only the rule says
  *why* — showing one without the other strips the explanation a human needs to act correctly.
- No page should be actioned on `gsc_avg_position < 1` without a manual raw-data check first —
  this is a known, unresolved data-quality gap, not a resolved edge case.
- No output here should be phrased as "will improve" or "will recover." Per the claim ladder
  (Section 2), this is decision-support only.

In [9]:
assert not any('auto' in c.lower() or 'apply' in c.lower() for c in ranked_queue.columns), \
    "queue contains a column that looks like an auto-action flag — playbook must stay human-gated"
print("Confirmed: no auto-action columns in the queue.")

Confirmed: no auto-action columns in the queue.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Monitoring has two clocks — pre-release and post-label.**

**Pre-release (implemented below, runs every cycle before the queue ships):** compares the new
month's eligible population size and rule action-mix against the previous month's. Run on Feb
vs. March, it did not come back clean: eligible pages grew **+24.1%** (93,871 → 116,512), and
the rule's action mix shifted — `content_fix` share rose **+6.3 points** while `monitor` fell
**−5.0 points**. Read together, more pages qualified for review at all, and a larger share of
them needed an engagement fix specifically. This check needs no outcome data — it would have
been visible the moment March's data landed, before a single page in the queue was reviewed.

**Post-label (implemented below, but only retroactively on data already in hand):** once next
month's outcomes are known, compare the realized decline base rate against the prior cycle's.
Applied to this playbook's own two adjacent windows, the base rate moved from 0.313 (Feb→Mar)
to 0.545 (Mar→Apr) — a +0.232 shift against a proposed ±0.15 alarm threshold. **The proposed
alarm would have fired.** This is not an isolated surprise: the pre-release check already showed
the action mix tilting toward `content_fix` before this number was knowable, so the two checks
tell a consistent story from two different points in time rather than two unrelated facts.

**What is and isn't real, stated plainly:**
- The population-health check and the retroactive base-rate check are both implemented and
  runnable, in this notebook, on historical data.
- The ±0.15 alarm threshold is a **proposed** policy value, not a tuned or validated one — it
  was chosen to be clearly smaller than the shift already observed, not derived from multiple
  cycles of data. With only two adjacent months available, there isn't yet a distribution of
  "normal" shifts to calibrate a threshold against.
- Neither check is wired into an automated production pipeline — both would need to be re-run
  by hand each cycle until that exists.
- The pre-release check flagging real movement (+24.1% population, a real action-mix shift) on
  the very first pair of months it was run on is itself a limitation worth naming: there's no
  evidence yet of what this check looks like on a *stable* month-pair, only on a shifting one.
- A monthly refit cadence (refitting `deployed_model` every cycle, as this notebook does)
  is a policy worth validating further, not a proven-optimal choice — it hasn't been compared
  against, say, refitting quarterly or on-trigger only.

In [10]:
def population_health_check(current_df, reference_df, current_name, reference_name):
    checks = {}
    checks['eligible_count'] = (len(current_df), len(reference_df))
    checks['eligible_count_pct_change'] = (len(current_df) - len(reference_df)) / len(reference_df)

    current_actions = current_df['action'].value_counts(normalize=True)
    reference_actions = reference_df['action'].value_counts(normalize=True)
    checks['action_mix_shift'] = (current_actions - reference_actions).round(3).to_dict()

    return checks

health = population_health_check(march_rule_df, feb_rule_df, 'March', 'Feb')
print(f"Eligible pages: March {health['eligible_count'][0]}, Feb {health['eligible_count'][1]}")
print(f"Change: {health['eligible_count_pct_change']:+.1%}")
print(f"Action-mix shift (share of pages, March minus Feb): {health['action_mix_shift']}")

Eligible pages: March 116512, Feb 93871
Change: +24.1%
Action-mix shift (share of pages, March minus Feb): {'snippet_fix': -0.013, 'monitor': -0.05, 'content_fix': 0.063}


In [11]:
PROPOSED_BASE_RATE_SHIFT_ALARM = 0.15  # proposed threshold, not yet a production system

feb_base_rate = feb_labelable['decline_label'].mean()
march_base_rate = labelable['decline_label'].mean()
observed_shift = march_base_rate - feb_base_rate

would_have_fired = abs(observed_shift) >= PROPOSED_BASE_RATE_SHIFT_ALARM

print(f"Feb->March base rate:  {feb_base_rate:.3f}")
print(f"March->April base rate: {march_base_rate:.3f}")
print(f"Observed shift: {observed_shift:+.3f}")
print(f"Proposed alarm threshold: ±{PROPOSED_BASE_RATE_SHIFT_ALARM}")
print(f"Would the proposed alarm have fired? {would_have_fired}")

Feb->March base rate:  0.313
March->April base rate: 0.545
Observed shift: +0.232
Proposed alarm threshold: ±0.15
Would the proposed alarm have fired? True


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.